# Skin Cancer Classification with ResNet50
## BADSA 2025 Capstone Project - Group 3
### Dataset: HAM10000
### Model: ResNet50 (Transfer Learning)

**Instructions:**
1. Upload your `data` folder to Google Drive in the path: `/content/drive/MyDrive/BADSA2025/`
2. Run all cells sequentially
3. Make sure GPU is enabled: Runtime → Change runtime type → GPU (T4)

## 1. Check GPU Availability

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("\nGPU Details:")
if tf.config.list_physical_devices('GPU'):
    print("✓ GPU is enabled!")
    gpu = tf.config.list_physical_devices('GPU')[0]
    print(f"Device: {gpu}")
else:
    print("⚠ No GPU found. Go to Runtime → Change runtime type → Hardware accelerator → GPU")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change to your project directory
import os
os.chdir('/content/drive/MyDrive/BADSA2025')
print(f"Current directory: {os.getcwd()}")
print("\nFiles in directory:")
!ls -la

## 3. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## 4. Configuration & Setup

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Training configuration
IMG_SIZE = 224  # ResNet50 standard input size
BATCH_SIZE = 32  # Increased from 16 since we have GPU now
EPOCHS = 20  # Increased from 15 - GPU can handle more
FINE_TUNE_EPOCHS = 10  # Increased from 5
LEARNING_RATE = 0.0001

# Class names
CLASS_NAMES = {
    'akiec': 'Actinic keratoses',
    'bcc': 'Basal cell carcinoma',
    'bkl': 'Benign keratosis-like lesions',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic nevi',
    'vasc': 'Vascular lesions'
}

print("=" * 60)
print("Skin Cancer Classification - ResNet50 Transfer Learning")
print("=" * 60)
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Fine-tune Epochs: {FINE_TUNE_EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")

## 5. Load and Preprocess Data

In [ ]:
# Load metadata
print("Loading metadata...")
metadata = pd.read_csv('data/HAM10000_metadata.csv')
print(f"Total samples: {len(metadata)}")
print(f"\nClass distribution:")
print(metadata['dx'].value_counts())

# Visualize class distribution
plt.figure(figsize=(10, 6))
metadata['dx'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Class Distribution in Dataset')
plt.xlabel('Disease Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Create image paths
def get_image_path(image_id):
    """Find image path in either part_1 or part_2 directory"""
    path1 = f'data/HAM10000_images_part_1/{image_id}.jpg'
    path2 = f'data/HAM10000_images_part_2/{image_id}.jpg'
    
    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    else:
        return None

metadata['path'] = metadata['image_id'].apply(get_image_path)
metadata = metadata.dropna(subset=['path'])
print(f"Valid images found: {len(metadata)}")

In [ ]:
# Split data into train/validation/test sets
print("Splitting data into train/validation/test sets...")
train_df, temp_df = train_test_split(
    metadata, test_size=0.3, random_state=42, stratify=metadata['dx']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['dx']
)

print(f"Training samples: {len(train_df)} (70%)")
print(f"Validation samples: {len(val_df)} (15%)")
print(f"Test samples: {len(test_df)} (15%)")

In [ ]:
# Calculate class weights for imbalanced dataset
class_weights_values = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_df['dx']),
    y=train_df['dx']
)
class_weights_dict = dict(enumerate(class_weights_values))
print("Class weights calculated:")
for idx, (class_name, weight) in enumerate(zip(sorted(train_df['dx'].unique()), class_weights_values)):
    print(f"  {class_name}: {weight:.2f}")

## 6. Data Augmentation & Generators

In [ ]:
# Data augmentation for training (ResNet50 preprocessing)
print("Setting up data augmentation with ResNet50 preprocessing...")
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ResNet50 specific preprocessing
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

# Only preprocessing for validation and test
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Create generators
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='path',
    y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Classes: {train_generator.class_indices}")

## 7. Build ResNet50 Model

In [ ]:
print("Building ResNet50 model with Transfer Learning...")

# Load pre-trained ResNet50 (without top classification layer)
base_model = ResNet50(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,  # Remove original classification head
    weights='imagenet'  # Use ImageNet pre-trained weights
)

# Freeze base model layers for initial transfer learning
base_model.trainable = False
print(f"Base model loaded: {len(base_model.layers)} layers frozen")

# Build complete model with custom classification head
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),  # Larger than MobileNetV2 (was 256)
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(len(CLASS_NAMES), activation='softmax')
])

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(),
        keras.metrics.Recall()
    ]
)

print("\nModel architecture:")
model.summary()

## 8. Setup Training Callbacks

In [ ]:
# Create models directory
os.makedirs('models', exist_ok=True)

# Setup callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        'models/best_resnet50_model.h5',
        monitor='val_recall',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured successfully!")

## 9. Train Model (Initial Phase)

In [ ]:
print("Starting initial training phase...")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print("\n" + "="*60)

# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

print("\nInitial training phase completed!")

## 10. Fine-Tuning (Unfreeze Layers)

In [ ]:
print("Starting fine-tuning phase...")
print("Unfreezing last 30 layers of ResNet50...\n")

# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Count trainable layers
trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Trainable layers in base model: {trainable_count}")

# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(),
        keras.metrics.Recall()
    ]
)

print(f"Learning rate reduced to: {LEARNING_RATE/10}")
print("\n" + "="*60)

# Continue training with fine-tuning
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

print("\nFine-tuning completed!")

## 11. Evaluate on Test Set

In [ ]:
print("Evaluating on test set...\n")
test_results = model.evaluate(test_generator, verbose=1)

print("\n" + "="*60)
print("TEST SET RESULTS (ResNet50)")
print("="*60)
print(f"Loss:      {test_results[0]:.4f}")
print(f"Accuracy:  {test_results[1]:.4f}")
print(f"AUC:       {test_results[2]:.4f}")
print(f"Precision: {test_results[3]:.4f}")
print(f"Recall:    {test_results[4]:.4f}")
print("="*60)

## 12. Save Model & Results

In [ ]:
# Save final model
model.save('models/final_resnet50_model.h5')
print("Model saved to 'models/final_resnet50_model.h5'")

# Save class names
import json
with open('models/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print("Class names saved to 'models/class_names.json'")

## 13. Visualization & Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# AUC
axes[1, 0].plot(history.history['auc'], label='Train AUC', linewidth=2)
axes[1, 0].plot(history.history['val_auc'], label='Val AUC', linewidth=2)
axes[1, 0].set_title('Model AUC', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
axes[1, 1].set_title('Model Recall', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/resnet50_training_history.png', dpi=150, bbox_inches='tight')
print("Training plots saved to 'models/resnet50_training_history.png'")
plt.show()

## 14. Training Complete!

Your ResNet50 model has been trained successfully on Google Colab's GPU!

**Next Steps:**
1. Download the trained model from `models/final_resnet50_model.h5`
2. Download class names from `models/class_names.json`
3. Use these files in your Streamlit app

**Model Performance:**
- Check the test results above
- Review the training history plots
- Compare with MobileNetV2 results

In [ ]:
# Optional: Download files directly from Colab
from google.colab import files

print("Downloading model files...")
files.download('models/final_resnet50_model.h5')
files.download('models/class_names.json')
files.download('models/resnet50_training_history.png')
print("\nDownload complete!")